# Stage-dependent capacity prediction in doped LNCM cathodes

Run all cells to verify the fitted models against the archived holdout predictions,
compare cross-validation and test metrics, and view the manuscript figures.
The notebook uses the deposited split, predictions and figure previews.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from sklearn.metrics import mean_squared_error, r2_score

data = pd.read_csv("ncm_dataset.csv", index_col="sample_index", float_precision="round_trip")
bundle = joblib.load("ncm_models.joblib")

# Translate manuscript headers to the names stored in the fitted estimators.
aliases = {"x_dopant": "dopant_M", "A_r,dopant": "Mr_dopant", "ΔIR": "△IR", "ΔEN": "△EN"}
X = data.rename(columns=aliases)[list(bundle["features"])]
test = data.query("dataset == 'Independent_test'").sort_values("split_order")
assert len(data) == 213 and len(test) == 43

## Model performance

Mean CV R² gives each of the five validation folds equal weight.
Pooled OOF R² uses all 170 validation predictions together.
Test RMSE is in mAh g⁻¹.

In [ ]:
rows = []
for (model, stage), fitted in bundle["models"].items():
    saved = bundle["predictions"][model, stage].set_index("sample_index")
    held_out = saved.loc[test.index]
    np.testing.assert_allclose(held_out.actual, test[stage], rtol=0, atol=1e-10)
    predicted = fitted.predict(X.loc[test.index])
    np.testing.assert_allclose(predicted, held_out.predicted, rtol=0, atol=1e-7)

    oof = saved.query("dataset == 'CV_validation'")
    fold_r2 = [r2_score(fold.actual, fold.predicted) for _, fold in oof.groupby("fold")]
    rows.append((model, stage, np.mean(fold_r2), r2_score(oof.actual, oof.predicted),
                 r2_score(held_out.actual, predicted),
                 mean_squared_error(held_out.actual, predicted) ** 0.5))

metrics = pd.DataFrame(rows, columns=["Model", "Stage", "Mean CV R²", "Pooled OOF R²",
                                     "Test R²", "Test RMSE"])
display(metrics.round(4))

## Figures

The previews below are rendered from the archived manuscript figures.
The full reproduction code is maintained in the repository.

In [ ]:
for number in ("2", "3", "4", "5", "S1a", "S1b", "S2"):
    display(Markdown(f"### Figure {number}"))
    display(Image(data=bundle["figures"][number], width=850))